# Minimal HEC Forward Pass In Colab

Qwen2-VL-2B, MNIST, batch size 1, and one hook that captures the last-token output of every LLM attention head.

In [ ]:
%pip install -q transformers torchvision pillow scikit-learn omegaconf matplotlib

In [ ]:
!rm -rf /content/HEC
!git clone -q https://github.com/AdhemarDeSenneville/HEC.git /content/HEC

In [ ]:
import sys

sys.path.insert(0, "/content/HEC")
from src.fewshot.classifier import ClassifierHEC_T, ClassifierHEC_V

In [ ]:
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from PIL import Image, ImageOps
from sklearn.decomposition import PCA
from torchvision.datasets import MNIST
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
CLASSES = [0, 1, 2, 3]
SHOTS = 5

## 1. Load 5 MNIST images per class

In [ ]:
mnist = MNIST(root="/content/data", train=False, download=True)

sample_ids = torch.cat([(mnist.targets == class_id).nonzero().flatten()[:SHOTS] for class_id in CLASSES]).tolist()
query_ids = torch.cat([(mnist.targets == class_id).nonzero().flatten()[SHOTS:SHOTS + 1] for class_id in CLASSES]).tolist()

images = [mnist[i][0].convert("RGB") for i in sample_ids]
query_images = [mnist[i][0].convert("RGB") for i in query_ids]

support_labels = torch.tensor([class_id for class_id in CLASSES for _ in range(SHOTS)], device="cuda")
query_labels = torch.tensor(CLASSES, device="cuda")

grid = Image.new("RGB", (SHOTS * 64, len(CLASSES) * 64), "white")
for i, image in enumerate(images):
    image = ImageOps.contain(image, (54, 54))
    grid.paste(image, ((i % SHOTS) * 64 + 5, (i // SHOTS) * 64 + 5))

query_grid = Image.new("RGB", (len(CLASSES) * 64, 64), "white")
for i, image in enumerate(query_images):
    image = ImageOps.contain(image, (54, 54))
    query_grid.paste(image, (i * 64 + 5, 5))

print(len(images), "support images")
display(grid)
print(len(query_images), "query images")
display(query_grid)

## 2. Load Qwen2-VL-2B

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
).to("cuda").eval()

## 3. Hook each LLM attention head

In [ ]:
def register_head_hooks(model):
    heads = {}
    handles = []

    for layer_id, layer in enumerate(model.model.language_model.layers):
        attn = layer.self_attn
        num_heads = attn.num_heads

        def save_heads(module, inputs, layer_id=layer_id, num_heads=num_heads):
            x = inputs[0]  # [batch, tokens, heads * head_dim], just before o_proj
            head_dim = x.shape[-1] // num_heads
            heads[layer_id] = x[:, -1, :].reshape(1, num_heads, head_dim).detach().float().cpu()

        handles.append(attn.o_proj.register_forward_pre_hook(save_heads))

    return heads, handles

## 4. Run one image at a time

In [ ]:
prompt = (
    "Classify this handwritten digit image. " # Domain Conditioning
    "Possible classes: digit 0, digit 1, digit 2, digit 3. " # + Class Conditioning
    "Answer with only one class name." # Force Next-Token Class Separability 
)

chat = processor.apply_chat_template(
    [{"role": "user", "content": [{"type": "image", "image": "Any"}, {"type": "text", "text": prompt}]}],
    tokenize=False,
    add_generation_prompt=True,
)

all_heads = []
for image in images:
    inputs = processor(text=[chat], images=[image], return_tensors="pt").to("cuda")

    heads, handles = register_head_hooks(model)
    with torch.no_grad():
        outputs = model(**inputs, use_cache=False)
    for handle in handles:
        handle.remove()

    all_heads.append(torch.stack([heads[i] for i in range(len(heads))], dim=1))

head_distribution = torch.cat(all_heads, dim=0)

all_query_heads = []
for image in query_images:
    inputs = processor(text=[chat], images=[image], return_tensors="pt").to("cuda")

    heads, handles = register_head_hooks(model)
    with torch.no_grad():
        outputs = model(**inputs, use_cache=False)
    for handle in handles:
        handle.remove()

    all_query_heads.append(torch.stack([heads[i] for i in range(len(heads))], dim=1))

query_head_distribution = torch.cat(all_query_heads, dim=0)

print("last logits:", tuple(outputs.logits.shape))
print("support head_distribution:", tuple(head_distribution.shape))
print("query_head_distribution:", tuple(query_head_distribution.shape))
print("shape meaning: [images, layers, heads, head_dim]")

## 5. Score and plot heads with HEC-V

In [ ]:
Ns, L, H, C = head_distribution.shape

hec_v = ClassifierHEC_V(k=20, tau=10, cov_reg=1, ensemble_method="mean_prob")
hec_v_output = hec_v.get_rank_scores(head_distribution.reshape(Ns, L * H, C).to("cuda"), support_labels)
head_scores = hec_v_output["head_scores"].reshape(L, H).cpu()

top_scores, top_idx = torch.topk(head_scores.flatten(), 20)
top_layers = (top_idx // H).numpy()
top_heads = (top_idx % H).numpy()

plt.figure(figsize=(10, 5))
plt.imshow(head_scores.numpy(), aspect="auto", cmap="viridis")
plt.scatter(top_heads, top_layers, s=140, facecolors="none", edgecolors="red", linewidths=2)
plt.colorbar(label="HEC-V head score")
plt.xlabel("head")
plt.ylabel("layer")
plt.title("Head scores, red contour = top 20")
plt.tight_layout()
plt.show()

sorted_scores = torch.sort(head_scores.flatten(), descending=True).values
plt.figure(figsize=(10, 4))
plt.plot(sorted_scores.numpy())
plt.scatter(range(20), sorted_scores[:20].numpy(), color="red", s=24)
plt.xlabel("head rank, best to worst")
plt.ylabel("HEC-V head score")
plt.title("Head scores from best to worst")
plt.tight_layout()
plt.show()

print("top 20 flat head ids:", top_idx.tolist())

## 6. Predict query images and show PCA for the best head

In [ ]:
Nq = query_head_distribution.shape[0]
top_idx_cuda = top_idx.to("cuda")
query_features = query_head_distribution.reshape(Nq, L * H, C).to("cuda")[:, top_idx_cuda, :]

query_probs, query_logits = hec_v.get_pred_labels(
    query_features,
    hec_v_output["head_means"][top_idx_cuda],
    hec_v_output["head_cov_inv"][top_idx_cuda],
    hec_v_output["log_prior"],
)
query_pred = hec_v_output["labels_unique"][query_probs.mean(dim=1).argmax(dim=-1)]

print("true:", query_labels.cpu().tolist())
print("pred:", query_pred.cpu().tolist())

best_idx = top_idx[0].item()
best_layer = best_idx // H
best_head = best_idx % H

pca_xy = PCA(n_components=2).fit_transform(
    torch.cat([
        head_distribution[:, best_layer, best_head, :],
        query_head_distribution[:, best_layer, best_head, :],
    ]).numpy()
)
support_xy = pca_xy[:Ns]
query_xy = pca_xy[Ns:]

support_y = support_labels.cpu().numpy()
query_y = query_labels.cpu().numpy()
colors = plt.cm.tab10(range(len(CLASSES)))

plt.figure(figsize=(6, 5))
for color, class_id in zip(colors, CLASSES):
    plt.scatter(support_xy[support_y == class_id, 0], support_xy[support_y == class_id, 1], color=color, label=f"digit {class_id}")
    plt.scatter(query_xy[query_y == class_id, 0], query_xy[query_y == class_id, 1], color=color, marker="x", s=140, linewidths=3)

plt.title(f"PCA of best head: layer {best_layer}, head {best_head}")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Build class text head embeddings

In [ ]:
class_prompts = [
    f"You are given an image of a digit {class_id}\n{prompt}"
    for class_id in CLASSES
]

class_chats = [
    processor.apply_chat_template(
        [{"role": "user", "content": [{"type": "text", "text": class_prompt}]}],
        tokenize=False,
        add_generation_prompt=True,
    )
    for class_prompt in class_prompts
]

all_text_heads = []
for class_chat in class_chats:
    inputs = processor(text=[class_chat], return_tensors="pt").to("cuda")

    heads, handles = register_head_hooks(model)
    with torch.no_grad():
        outputs = model(**inputs, use_cache=False)
    for handle in handles:
        handle.remove()

    all_text_heads.append(torch.stack([heads[i] for i in range(len(heads))], dim=1))

class_head_distribution = torch.cat(all_text_heads, dim=0)

print("class_head_distribution:", tuple(class_head_distribution.shape))
print("shape meaning: [classes, layers, heads, head_dim]")

## 8. Score heads with HEC-T

In [ ]:
K = len(CLASSES)
text_features = class_head_distribution.permute(1, 2, 0, 3).reshape(L * H, K, C).to("cuda")

hec_t = ClassifierHEC_T(k=20, tau=1, ensemble_method="mean_prob", encoding="direct")
hec_t_output = hec_t.get_rank_scores(
    head_distribution.reshape(Ns, L * H, C).to("cuda"),
    text_features,
    support_labels,
)
text_head_scores = hec_t_output["head_scores"].reshape(L, H).cpu()

text_top_scores, text_top_idx = torch.topk(text_head_scores.flatten(), 20)
text_top_layers = (text_top_idx // H).numpy()
text_top_heads = (text_top_idx % H).numpy()

plt.figure(figsize=(10, 5))
plt.imshow(text_head_scores.numpy(), aspect="auto", cmap="viridis")
plt.scatter(text_top_heads, text_top_layers, s=140, facecolors="none", edgecolors="red", linewidths=2)
plt.colorbar(label="HEC-T head score")
plt.xlabel("head")
plt.ylabel("layer")
plt.title("HEC-T head scores, red contour = top 20")
plt.tight_layout()
plt.show()

text_sorted_scores = torch.sort(text_head_scores.flatten(), descending=True).values
plt.figure(figsize=(10, 4))
plt.plot(text_sorted_scores.numpy())
plt.scatter(range(20), text_sorted_scores[:20].numpy(), color="red", s=24)
plt.xlabel("head rank, best to worst")
plt.ylabel("HEC-T head score")
plt.title("HEC-T head scores from best to worst")
plt.tight_layout()
plt.show()

print("HEC-T top 20 flat head ids:", text_top_idx.tolist())

## 9. Predict query images with HEC-T and show PCA

In [ ]:
text_top_idx_cuda = text_top_idx.to("cuda")
query_text_features = query_head_distribution.reshape(Nq, L * H, C).to("cuda")[:, text_top_idx_cuda, :]

query_text_probs = hec_t.get_pred_labels(
    query_text_features,
    text_features[text_top_idx_cuda],
)
query_text_pred = hec_t_output["labels_unique"][query_text_probs.mean(dim=1).argmax(dim=-1)]

print("true:", query_labels.cpu().tolist())
print("HEC-T pred:", query_text_pred.cpu().tolist())
print("HEC-T accuracy:", (query_text_pred == query_labels).float().mean().item())

text_best_idx = text_top_idx[0].item()
text_best_layer = text_best_idx // H
text_best_head = text_best_idx % H

text_pca_xy = PCA(n_components=2).fit_transform(
    torch.cat([
        head_distribution[:, text_best_layer, text_best_head, :],
        query_head_distribution[:, text_best_layer, text_best_head, :],
        class_head_distribution[:, text_best_layer, text_best_head, :],
    ]).numpy()
)
support_xy = text_pca_xy[:Ns]
query_xy = text_pca_xy[Ns:Ns + Nq]
class_xy = text_pca_xy[Ns + Nq:]

support_y = support_labels.cpu().numpy()
query_y = query_labels.cpu().numpy()
colors = plt.cm.tab10(range(len(CLASSES)))

plt.figure(figsize=(6, 5))
for color, class_id in zip(colors, CLASSES):
    plt.scatter(support_xy[support_y == class_id, 0], support_xy[support_y == class_id, 1], color=color, label=f"digit {class_id} support")
    plt.scatter(query_xy[query_y == class_id, 0], query_xy[query_y == class_id, 1], color=color, marker="x", s=140, linewidths=3, label=f"digit {class_id} query")
    plt.scatter(class_xy[class_id, 0], class_xy[class_id, 1], color=color, marker="*", s=220, edgecolors="black", label=f"digit {class_id} text")

plt.title(f"HEC-T PCA best head: layer {text_best_layer}, head {text_best_head}")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Class / split", fontsize=8)
plt.tight_layout()
plt.show()